In [3]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Fetch data manually via URL to bypass Hugging Face URI restrictions
print("Loading online Twitter dataset...")
dataset_url = "https://raw.githubusercontent.com/satyamk28/Twitter-Sentiment-Analysis/master/twitter_training.csv"
df = pd.read_csv(dataset_url, names=['id', 'entity', 'sentiment', 'text'])

# Filter and structure the text columns
df = df.dropna(subset=['text', 'sentiment'])
df = df[df['sentiment'].str.lower().isin(['positive', 'negative'])] # Binary for simple classification
df['label'] = df['sentiment'].str.lower().map({'negative': 0, 'positive': 1})

# Sample a tiny fraction (100 train, 50 evaluation) for rapid learning execution
train_df = df.sample(n=100, random_state=42)
eval_df = df.sample(n=50, random_state=24)

# Convert pandas DataFrames directly into Hugging Face Dataset structures
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# 2. Load the Model and Tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # Ensure text field is parsed as string context
    text_list = [str(t) for t in examples["text"]]
    return tokenizer(text_list, truncation=True, padding="max_length", max_length=64)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 3. Initialize Model with 2 output classes (Negative, Positive)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

# 4. Define Training parameters
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

# 5. Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# 6. Run Fine-Tuning
print("\nStarting Fine-Tuning workflow...")
trainer.train()

# 7. Evaluate Performance
print("\nRunning model evaluation metrics...")
evaluation_results = trainer.evaluate()
print(f"\nFinal Test Accuracy: {evaluation_results['eval_accuracy']:.4f}")
print(f"Final Test F1-Score: {evaluation_results['eval_f1']:.4f}")

Loading online Twitter dataset...
Tokenizing datasets...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Running model evaluation metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.685255,0.692870,2,0.480000,0.427029



Final Test Accuracy: 0.4800
Final Test F1-Score: 0.4270
